# Detecting honeybee wingbeat buzz in microphone recordings

**Approach**

1. Load the *background-only* recordings (HVAC fan etc.) and learn, for every frequency bin, what "normal" looks like (robust median and spread of the log-power spectrogram).
2. For any new recording, convert its spectrogram into **z-scores relative to that background**, so HVAC tones and rumble are automatically down-weighted.
3. Honeybee flight sound is a **harmonic series**: a fundamental of roughly 200-300 Hz (it varies with temperature and activity) plus integer multiples. For each frame, search a grid of candidate fundamentals f0 and score how far above background the energy sits at f0, 2·f0, 3·f0, ... The best-scoring f0 gives a per-frame detection score.
4. Choose the detection threshold from the background itself (a chosen false-alarm rate on held-out background frames), then clean up with morphological smoothing so only sustained buzzes are reported as events.
5. Check sensitivity by injecting synthetic buzz into background recordings at different levels.

Everything tunable lives in the config cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
from pathlib import Path
from scipy.ndimage import maximum_filter1d, binary_closing, binary_opening, label

# ---- paths ----
BG_DIR = Path("recordings/background")   # recordings with NO bees (HVAC etc.)
TEST_DIR = Path("recordings/test")       # recordings to scan for buzzing
AUDIO_GLOB = "*.mp3"

# ---- audio / STFT ----
SR = 8000          # resample target (Hz). Wingbeat harmonics live well below 4 kHz
N_FFT = 2048       # 256 ms window -> 3.9 Hz bins
HOP = 512          # 64 ms hop

# ---- bee model ----
F0_RANGE = (180, 320)   # Hz, candidate wingbeat fundamental
F0_STEP = 2.0           # Hz, spacing of the f0 search grid
N_HARMONICS = 4         # how many harmonics (f0, 2*f0, ...) to include in the score
TOL_BINS = 2            # +/- frequency-bin tolerance around each harmonic
BAND = (100, 1500)      # Hz, used to estimate and remove broadband (non-harmonic) excess

# ---- decision / cleanup ----
FA_PERCENTILE = 99.5    # threshold = this percentile of background scores
MIN_DUR_S = 0.5         # discard detections shorter than this
MAX_GAP_S = 0.3         # bridge gaps shorter than this inside one buzz

## Helpers

`librosa.load` handles the mp3 decoding (it uses `soundfile` or `audioread`/ffmpeg depending on your install) and resamples to `SR`. `power_to_db` with `top_db=None` keeps absolute dB values, which matters here because we compare different files against a common background model.

In [ ]:
freqs = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
bin_hz = freqs[1] - freqs[0]
band_mask = (freqs >= BAND[0]) & (freqs <= BAND[1])


def load_audio(path):
    y, _ = librosa.load(path, sr=SR, mono=True)
    return y


def log_spec(y):
    """Power spectrogram in dB, shape (n_bins, n_frames)."""
    S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)) ** 2
    return librosa.power_to_db(S, ref=1.0, top_db=None)


def list_audio(d):
    return sorted(Path(d).glob(AUDIO_GLOB))


def frame_times(n_frames):
    return librosa.frames_to_time(np.arange(n_frames), sr=SR, hop_length=HOP)

## 1. Characterise the background

For each frequency bin we store the median and a robust standard deviation (1.4826 × MAD) of the dB level across all background frames. Robust statistics stop the occasional door slam or cough in a "clean" recording from distorting the model.

If you have 3+ background files, ~30% are held out to calibrate the threshold honestly; otherwise all frames are used for both (slightly optimistic, and the notebook will say so).

In [ ]:
bg_files = list_audio(BG_DIR)
assert bg_files, f"No {AUDIO_GLOB} files found in {BG_DIR}"

bg_audio = {f.name: load_audio(f) for f in bg_files}
bg_specs = {name: log_spec(y) for name, y in bg_audio.items()}

if len(bg_files) >= 3:
    n_train = max(1, int(round(0.7 * len(bg_files))))
    train_names = [f.name for f in bg_files[:n_train]]
    val_names = [f.name for f in bg_files[n_train:]]
else:
    train_names = val_names = [f.name for f in bg_files]
    print("Fewer than 3 background files: calibrating on the training data itself.")

train = np.concatenate([bg_specs[n] for n in train_names], axis=1)
mu = np.median(train, axis=1)
sd = 1.4826 * np.median(np.abs(train - mu[:, None]), axis=1)
sd = np.maximum(sd, 0.5)   # floor (dB) so near-silent bins don't explode the z-scores

print(f"{len(train_names)} training file(s), {len(val_names)} validation file(s), "
      f"{train.shape[1]} training frames")

fig, ax = plt.subplots(figsize=(10, 3.5))
m = freqs <= 1500
ax.plot(freqs[m], mu[m], label="background median")
ax.fill_between(freqs[m], (mu - 3 * sd)[m], (mu + 3 * sd)[m], alpha=0.3, label="±3 robust SD")
ax.set(xlabel="Frequency (Hz)", ylabel="Power (dB)", title="Background model")
ax.axvspan(*F0_RANGE, color="orange", alpha=0.15, label="f0 search range")
ax.legend()
plt.show()

## 2. Harmonic-comb detection score

Steps per frame:

1. **z-score** each bin against the background model.
2. **Subtract the frame's median z over `BAND`.** A click or broadband burst raises every bin and is removed here; a bee buzz only raises narrow peaks, so the median barely moves.
3. Take a **local max** across ±`TOL_BINS` so slight pitch drift or bin misalignment doesn't hurt.
4. For every candidate f0, average that value over f0, 2·f0, … (`N_HARMONICS`). The score is the maximum over f0, and the argmax is the estimated wingbeat frequency.

In [ ]:
f0_grid = np.arange(F0_RANGE[0], F0_RANGE[1] + F0_STEP, F0_STEP)
# bin index of each harmonic for each candidate f0: shape (n_f0, N_HARMONICS)
harm_idx = np.round(np.outer(f0_grid, np.arange(1, N_HARMONICS + 1)) / bin_hz).astype(int)
assert harm_idx.max() < len(freqs), "Harmonics exceed Nyquist; raise SR or lower N_HARMONICS"


def zscore(L):
    z = (L - mu[:, None]) / sd[:, None]
    z -= np.median(z[band_mask], axis=0, keepdims=True)   # remove broadband excess
    return z


def harmonic_score(z):
    """Return (score, best_f0) per frame."""
    zmax = maximum_filter1d(z, size=2 * TOL_BINS + 1, axis=0)
    per_f0 = zmax[harm_idx].mean(axis=1)            # (n_f0, n_frames)
    best = per_f0.argmax(axis=0)
    return per_f0.max(axis=0), f0_grid[best]


# Calibrate the threshold on held-out background frames
val_scores = np.concatenate([harmonic_score(zscore(bg_specs[n]))[0] for n in val_names])
THRESH = np.percentile(val_scores, FA_PERCENTILE)
print(f"Background score: median {np.median(val_scores):.2f}, "
      f"{FA_PERCENTILE}th percentile {THRESH:.2f}  ->  THRESH = {THRESH:.2f}")

plt.figure(figsize=(6, 3))
plt.hist(val_scores, bins=80)
plt.axvline(THRESH, color="r", ls="--", label="threshold")
plt.xlabel("Harmonic score on background"); plt.ylabel("frames"); plt.legend()
plt.show()

## 3. Detector

Frame-level hits are cleaned up with a morphological *closing* (bridges short dropouts within a buzz) followed by an *opening* (removes runs shorter than `MIN_DUR_S`). Signals are edge-padded first so detections touching the start or end of a file aren't eroded.

In [ ]:
def clean_mask(hit):
    k_gap = max(1, int(np.ceil(MAX_GAP_S * SR / HOP)))
    k_min = max(1, int(np.ceil(MIN_DUR_S * SR / HOP)))
    pad = max(k_gap, k_min)
    h = np.pad(hit, pad, mode="edge")
    h = binary_closing(h, structure=np.ones(k_gap, bool))
    h = binary_opening(h, structure=np.ones(k_min, bool))
    return h[pad:-pad]


def detect(y, thresh=None):
    thresh = THRESH if thresh is None else thresh
    L = log_spec(y)
    z = zscore(L)
    score, f0 = harmonic_score(z)
    hit = clean_mask(score > thresh)
    t = frame_times(len(score))

    lab, n = label(hit)
    rows = []
    for i in range(1, n + 1):
        idx = np.where(lab == i)[0]
        rows.append(dict(start_s=t[idx[0]], end_s=t[idx[-1]] + HOP / SR,
                         f0_hz=np.median(f0[idx]), mean_score=score[idx].mean()))
    events = pd.DataFrame(rows, columns=["start_s", "end_s", "f0_hz", "mean_score"])
    return dict(L=L, t=t, score=score, f0=f0, hit=hit, events=events)


def plot_detection(res, title="", fmax=1500):
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                                 gridspec_kw=dict(height_ratios=[2, 1]))
    librosa.display.specshow(res["L"], sr=SR, hop_length=HOP, x_axis="time", y_axis="hz", ax=a1)
    a1.set_ylim(0, fmax); a1.set_title(title)
    a2.plot(res["t"], res["score"], lw=0.8, label="harmonic score")
    a2.axhline(THRESH, color="r", ls="--", label="threshold")
    for _, e in res["events"].iterrows():
        for a in (a1, a2):
            a.axvspan(e.start_s, e.end_s, color="lime", alpha=0.25)
    a2.set(xlabel="Time (s)", ylabel="score"); a2.legend(loc="upper right")
    plt.tight_layout(); plt.show()

## 4. Scan the test recordings

In [ ]:
test_files = list_audio(TEST_DIR)
all_events = []
for f in test_files:
    res = detect(load_audio(f))
    ev = res["events"].assign(file=f.name)
    all_events.append(ev)
    frac = res["hit"].mean()
    print(f"{f.name}: {len(ev)} event(s), {100 * frac:.1f}% of frames flagged")
    plot_detection(res, title=f.name)

events_df = pd.concat(all_events, ignore_index=True) if all_events else pd.DataFrame()
events_df

## 5. Sensitivity check with synthetic buzz

You probably don't have ground-truth labels yet, so this injects a synthetic harmonic "bee" (slowly wandering f0 around 240 Hz, 1/k harmonic amplitudes) into a held-out background recording at several levels and measures how many frames inside the injected segment are flagged, versus false alarms outside it.

SNR here is buzz RMS relative to the **full-band** RMS of the background. HVAC noise is usually dominated by low-frequency rumble, so this is a conservative number: the *in-band* SNR at the buzz harmonics is much better.

In [ ]:
def synth_buzz(n, f0=240.0, wobble=0.03, n_harm=6, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(n) / SR
    f = f0 * (1 + wobble * np.sin(2 * np.pi * 0.7 * t + rng.uniform(0, 6.28)))
    phase = 2 * np.pi * np.cumsum(f) / SR
    y = sum(np.sin(k * phase) / k for k in range(1, n_harm + 1))
    return y / np.sqrt(np.mean(y ** 2))


def inject(y_bg, snr_db, start_s=None, dur_s=3.0):
    n = int(dur_s * SR)
    start = int((len(y_bg) / SR / 2 - dur_s / 2) * SR) if start_s is None else int(start_s * SR)
    y = y_bg.copy()
    amp = np.sqrt(np.mean(y_bg ** 2)) * 10 ** (snr_db / 20)
    y[start:start + n] += amp * synth_buzz(n)
    truth = np.zeros(len(frame_times(1 + len(y) // HOP)), bool)
    truth[start // HOP: (start + n) // HOP + 1] = True
    return y, truth


snrs = np.arange(-40, 1, 4)
recall, false_alarm = [], []
for snr in snrs:
    r, fa = [], []
    for name in val_names:
        y, truth = inject(bg_audio[name], snr)
        res = detect(y)
        truth = truth[:len(res["hit"])]
        r.append(res["hit"][truth].mean())
        fa.append(res["hit"][~truth].mean())
    recall.append(np.mean(r)); false_alarm.append(np.mean(fa))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(snrs, recall, "o-", label="frames detected inside buzz")
ax.plot(snrs, false_alarm, "s--", label="frames flagged outside buzz")
ax.set(xlabel="Buzz RMS / background RMS (dB, full band)", ylabel="fraction of frames",
       title="Detection sensitivity (synthetic buzz)")
ax.legend(); plt.show()

## Tuning notes

- **Too many false alarms in real recordings?** Raise `FA_PERCENTILE` (e.g. 99.9), increase `MIN_DUR_S`, or add more/more varied background recordings (different HVAC states, time of day). A held-out background set is the most honest way to set the threshold.
- **Missing weak buzzing?** Lower `FA_PERCENTILE`, increase `N_HARMONICS` if your recordings show clear higher harmonics, or widen `F0_RANGE` if bees were warm/cold (wingbeat frequency shifts with temperature).
- **Look at a spectrogram of a known buzz** first and check that the harmonic stack falls where you expect; adjust `F0_RANGE` accordingly. Nothing here is specific to *Apis mellifera* beyond those frequency ranges.
- **Other sounds with harmonics** (a motor whose speed changes, people talking or whistling, other insects) can trigger the detector. If that matters, add recordings of those sounds to the background set, or tighten `F0_RANGE`.
- **Next steps if you want more performance:** label some real buzz segments and train a small classifier (e.g. logistic regression or gradient boosting) on features like the score, f0, harmonic ratios and spectral flux, or use MFCCs/log-mel frames with a simple model.